# 实验四 · 白平衡 Gray World（读交织 → 统计 → 增益 → 写回）

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐⭐⭐ 综合　|　**预计时长**：30–40 分钟

> **实验说明**
> 1. 本实验为**综合案例**，将前几个实验的技术组合于一个真实的图像处理算法之中。采用分步实现的方式：由 **v1 全串行版本**起，依次将两遍处理改用 **v2 NEON 实现**，并在应用阶段引入 **v3 循环展开**。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 三个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方法，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。
> 6. 本实验综合运用了**实验二的规约**、**实验三的结构化访存**，以及**定点运算与饱和处理**，建议先完成前述实验。

## 🎯 学习目标

完成本实验后，学生应能够：

- 理解**数据相关（data-dependent）** 算法的含义：增益由图像自身的统计量决定，而非预先设定的常数
- 掌握 **Gray World（灰度世界）** 假设及其**两遍法**（第一遍统计、第二遍应用）
- 将**规约**（第一遍的通道求和，对应实验二）应用于真实图像数据
- 理解 **Q8 定点表示**的原理与动机，以及为何需要对增益进行**范围钳制**
- 综合运用 **`vld3`/`vst3` 结构化访存**、**加宽运算防溢出**、**定点乘法**与**饱和窄化**
- 分析**两遍法**（图像被读取两次）对性能的影响，并学会区分某一遍是卡在**访存**上还是卡在**指令吞吐**上

## 🗺️ 学习路径

1. **准备阶段**：理解白平衡要解决的问题（色偏校正）与 Gray World 假设
2. **v1 · 全串行基准**：实现 `gray_world_no_vec`（关闭向量化，作为基准）与 `gray_world_serial`（允许自动向量化）
   → 建立性能基准，并观察程序算出的数据相关增益
3. **v2 · NEON 两遍实现**：新增 `gray_world_neon`，第一遍用 NEON 规约统计、第二遍用 NEON 应用增益
   → 掌握规约统计与“解交织—计算—交织”的应用过程
4. **v3 · 应用阶段循环展开**：新增 `gray_world_unroll`
   → 考察循环展开对应用阶段的效果
5. **可视化与分析**：以 v3 的输出结果绘制加速比柱状图，分析两遍法的访存特征

## 1. 背景与动机

相机在不同光源下拍摄的图像可能出现整体**偏色**（如偏蓝、偏黄）。**白平衡**的目的即是将颜色校正回中性。

最经典的 **Gray World（灰度世界）** 算法基于如下假设：*一幅自然场景图像中 R、G、B 三通道的平均值应当趋于相等（即整体呈灰）*。据此，算法先**统计**每个通道的平均值，再为每个通道乘以一个增益，使三通道的均值被拉平。

> ⚠️ **关键**：增益**必须由图像的统计量计算得出**（数据相关），预先写死的固定增益不构成真正的白平衡。

## 2. 算法与公式

**第一遍（规约）**：统计各通道之和 $\Sigma_R,\Sigma_G,\Sigma_B$，得到均值 $avg_c=\Sigma_c/N$。

**计算增益**（Gray World）：
$$gain_c = \frac{grayAvg}{avg_c} = \frac{\Sigma_R+\Sigma_G+\Sigma_B}{3\,\Sigma_c}$$
该增益以 **Q8 定点**表示，并将其钳制于 $[0.25\times, 4\times]$（详见下节说明）。

**第二遍（应用）**：$Out_c = \mathrm{sat}(c \times gain_c)$，即解交织 → 各通道乘以对应增益 → 饱和处理 → 交织写回。

> 说明：增益公式中的 $N$（像素数）在分子与分母中相消，因此可直接由通道之和计算增益，无需先做除法。

## 2.1 Q8 定点表示与增益钳制（重点）

本节详细说明两个关键设计：为什么用 **Q8 定点**表示增益，以及为什么要将增益**钳制**于 $[0.25\times, 4\times]$。

### （1）如果不用定点：直接用浮点
最直接的做法是用**浮点数**保存增益并逐像素相乘：
```c
float gain_r = grayAvg / avg_r;          // 例如 1.6
out = (uint8_t)(pixel * gain_r + 0.5f);  // 浮点乘法 + 四舍五入
```
这在功能上正确，但在**边缘端/嵌入式设备**上存在两个问题：
- **依赖浮点单元（FPU）**：部分低功耗 ARM 核心的 FPU 面积大、功耗高，甚至精简核心不含 FPU；浮点运算延迟也较高。
- **SIMD 并行度低**：128 位向量只能容纳 **4 个 32 位浮点**，而图像像素本是 8 位整数（u8），用浮点处理无法发挥 SIMD 对整数的并行优势。

### （2）改用定点：Q8 表示
**定点数**用整数来表示小数：**Q8** 约定将一个数乘以 $2^8=256$ 后存为整数，即“整数的低 8 位代表小数部分”。

以增益 $1.6$ 为例：
$$gain_{q8} = \mathrm{round}(1.6 \times 256) = 410,\qquad \text{其真实值} = 410 / 256 \approx 1.602$$
相乘时用**整数乘法加移位**代替浮点乘法：
```c
// out = round(pixel * gain) = (pixel * gain_q8 + 128) >> 8
uint32_t v = ((uint32_t)pixel * gain_q8 + 128u) >> 8;  // +128 实现四舍五入
```
其中 `>> 8` 等价于除以 256（还原 Q8），`+128` 是在移位前加“半个单位”（256 的一半）以实现四舍五入，而非直接截断。

**采用 Q8 定点的原因**：
- **避开 FPU**：仅用整数 ALU 的乘法与移位即可完成，省电、省面积、延迟低；
- **SIMD 并行度高**：128 位向量可容纳 **8 个 16 位整数**，与 8 位图像数据相匹配，充分利用并行；
- **结果可复现**：整数运算无浮点舍入的不确定性，便于逐位校验（`PASS`/`FAIL`）。

> 之所以取 **8** 位小数（Q8），是因为像素为 8 位（0–255），刻度 256 与像素范围天然匹配；其精度为 $1/256 \approx 0.4\%$，肉眼无法察觉，同时不会使中间结果过度膨胀，是精度与范围的平衡点。

### （3）为什么必须钳制增益范围
Gray World 的增益是**由图像统计量算出**的（$gain_c = grayAvg / avg_c$），其取值可能过大或过小，若不加限制会引发问题：

- **某通道过暗时，增益急剧增大。** 若某通道均值 $avg_c$ 很小，则增益很大。过度放大一个暗通道会带来两个后果：一是**放大噪声**（暗部信噪比低，放大后噪点明显，画质下降）；二是**超出计算范围**（在定点实现中，增益是传入 `vmull_n_u16` 的 16 位标量，若过大会超出 u16 上限而出错）。
- **某通道过亮时，增益过小。** 会将该通道压得过暗，导致矫正过度。

因此，真实相机 ISP 中的白平衡增益**总是带钳位**。本实验取工程上常用的区间 $[0.25\times, 4\times]$，即“每个通道最多放大 4 倍、最多压暗至 1/4”。

### （4）钳制范围在两种表示下的等价关系
钳制的是**增益的真实倍数**，与用何种数值表示无关。二者的对应关系为：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">表示方式</th>
      <th style="text-align: left;">下限（0.25×）</th>
      <th style="text-align: left;">上限（4×）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>浮点</strong>（直接倍数）</td>
      <td style="text-align: left;"><code>0.25f</code></td>
      <td style="text-align: left;"><code>4.0f</code></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>Q8 定点</strong>（×256 后的整数）</td>
      <td style="text-align: left;"><code>0.25 × 256 = 64</code></td>
      <td style="text-align: left;"><code>4 × 256 = 1024</code></td>
    </tr>
  </tbody>
</table>

也就是说：**若采用浮点实现，应将增益钳制于 $[0.25, 4.0]$；本实验采用 Q8 定点，等价地将其钳制于整数区间 $[64, 1024]$。** 代码中即为：
```c
#define GAIN_MIN 64u    // 0.25x in Q8
#define GAIN_MAX 1024u  // 4.00x in Q8
```
此外，上限 `1024` 也确保增益不超过 `vmull_n_u16` 所要求的 16 位标量范围（$1024 < 65535$），从而保证定点乘法的正确性。

## 3. 核心 NEON 指令与技巧

**第一遍 · 规约统计**（对应实验二）：
```c
uint32x4_t acc_r=vdupq_n_u32(0), acc_g=..., acc_b=...;
for (...) {
  uint8x16x3_t px = vld3q_u8(rgb + i*3);
  acc_r = vpadalq_u16(acc_r, vpaddlq_u8(px.val[0]));  // 加宽成对相加并累加
  acc_g = vpadalq_u16(acc_g, vpaddlq_u8(px.val[1]));
  acc_b = vpadalq_u16(acc_b, vpaddlq_u8(px.val[2]));
}
sum_r = vaddvq_u32(acc_r);   // 水平规约为标量（大图像需定期累加到 u64 防溢出）
```
**第二遍 · 应用增益**（解交织—计算—交织）：
```c
// 单通道 ×gain：u8 →vmovl→ u16 →vmull_n→ u32 →vrshrn(>>8)→ u16 →vqmovn(饱和)→ u8
uint8x16x3_t px = vld3q_u8(rgb + i*3);
px.val[0] = gain_u8x16(px.val[0], gain_r); ...
vst3q_u8(out + i*3, px);
```

- **`vpaddlq_u8` / `vpadalq_u16`**：逐级加宽累加，防止 u8/u16 累加溢出
- **`vmull_n_u16`**：u16 与标量增益相乘得到 u32
- **`vrshrn` / `vqmovn`**：带舍入的右移窄化，以及**饱和**窄化（将超出 255 的值钳制为 255，无需分支）

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 校验 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows


In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# 创建源代码目录
!mkdir -p src_whitebalance

## 5. v1 · 全串行基准实现

第一个版本包含**两个完整的串行 Gray World 实现**，区别仅在于是否允许编译器自动向量化：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>gray_world_no_vec</code></td>
      <td style="text-align: left;">全程使用串行的统计（<code>sums_serial</code>）与应用（<code>apply_serial</code>），并<strong>关闭自动向量化</strong>，作为<strong>性能基准</strong>（1.00×）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>gray_world_serial</code></td>
      <td style="text-align: left;">相同的串行实现，但<strong>允许编译器自动向量化</strong></td>
    </tr>
  </tbody>
</table>

> 说明：由于本案例涉及规约统计、结构化访存与饱和处理等多种运算，各版本共享同一套底层函数（`sums_serial`/`sums_neon`、`apply_serial`/`apply_neon`/`apply_neon_unroll`、`gains_from_sums` 等）；不同版本的区别在于**顶层函数调用了其中的串行实现还是 NEON 实现**。因此三个源文件的函数体基本相同，差异体现在 `main` 中所测量的实现上。

### 观察数据相关增益
程序采用一幅**偏蓝**的测试图像，并在运行时打印其统计出的各通道均值与计算出的增益。偏蓝图像应得到“R 通道增益较大、B 通道增益较小”的结果——这表明增益确实是**由数据算出**的，而非预先设定。

### 数据布局
- `rgb`：交织存储的输入图像；`out`：交织存储的输出图像（非原地，无需复位）
- 参考结果 `o_ref` 由 `gray_world_no_vec` 生成；`check_diff` 容差取 1（允许定点舍入带来的 ±1 差异）

In [ ]:
%%writefile src_whitebalance/whitebalance_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Gray-World gain clamp range in Q8 fixed-point
#define GAIN_MIN 64u    // 0.25x in Q8
#define GAIN_MAX 1024u  // 4.00x in Q8

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference (allow +/-1 rounding difference)
static const char* check_diff(const uint8_t* ref, const uint8_t* test,
                              long bytes) {
  int max_diff = 0;
  for (long i = 0; i < bytes; i++) {
    int d = abs((int)ref[i] - (int)test[i]);
    if (d > max_diff) max_diff = d;
  }
  if (max_diff <= 1)
    return "PASS";
  else
    return "FAIL";
}

// Compute Gray-World gains (Q8) from channel sums, then clamp to [0.25x, 4x]
static inline void gains_from_sums(uint64_t sum_r, uint64_t sum_g,
                                   uint64_t sum_b, uint32_t* gain_r,
                                   uint32_t* gain_g, uint32_t* gain_b) {
  uint64_t total = sum_r + sum_g + sum_b;
  uint32_t r = sum_r ? (uint32_t)((256ULL * total) / (3ULL * sum_r)) : 256u;
  uint32_t g = sum_g ? (uint32_t)((256ULL * total) / (3ULL * sum_g)) : 256u;
  uint32_t b = sum_b ? (uint32_t)((256ULL * total) / (3ULL * sum_b)) : 256u;
  *gain_r = r < GAIN_MIN ? GAIN_MIN : (r > GAIN_MAX ? GAIN_MAX : r);
  *gain_g = g < GAIN_MIN ? GAIN_MIN : (g > GAIN_MAX ? GAIN_MAX : g);
  *gain_b = b < GAIN_MIN ? GAIN_MIN : (b > GAIN_MAX ? GAIN_MAX : b);
}

// Apply gain to one channel sample: (c * gain_q8 + 128) >> 8, clamp to [0,255]
static inline uint8_t wb_px(uint8_t c, uint32_t gain_q8) {
  uint32_t v = ((uint32_t)c * gain_q8 + 128u) >> 8;
  return v > 255u ? 255u : (uint8_t)v;
}

// Pass 1 (serial): sum each channel over the image
static inline void sums_serial(const uint8_t* restrict rgb, long n,
                               uint64_t* sum_r, uint64_t* sum_g,
                               uint64_t* sum_b) {
  uint64_t r = 0, g = 0, b = 0;
  for (long i = 0; i < n; i++) {
    r += rgb[3 * i + 0];
    g += rgb[3 * i + 1];
    b += rgb[3 * i + 2];
  }
  *sum_r = r;
  *sum_g = g;
  *sum_b = b;
}

// Pass 2 (serial): apply per-channel gains
static inline void apply_serial(const uint8_t* restrict rgb,
                                uint8_t* restrict out, long n, uint32_t gain_r,
                                uint32_t gain_g, uint32_t gain_b) {
  for (long i = 0; i < n; i++) {
    out[3 * i + 0] = wb_px(rgb[3 * i + 0], gain_r);
    out[3 * i + 1] = wb_px(rgb[3 * i + 1], gain_g);
    out[3 * i + 2] = wb_px(rgb[3 * i + 2], gain_b);
  }
}

// Full pipeline: Serial statistics + Serial apply (compiler may auto-vectorize)
void gray_world_serial(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_serial(rgb, out, n, gain_r, gain_g, gain_b);
}

// Full pipeline: Serial statistics + Serial apply (vectorization disabled) -
// Baseline
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gray_world_no_vec(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_serial(rgb, out, n, gain_r, gain_g, gain_b);
}

// Benchmark helper: run kernel NTIMES, keep the best (lowest) time
typedef void (*wb_fn)(const uint8_t*, uint8_t*, long);

static double bench(wb_fn fn, const uint8_t* rgb, uint8_t* out, long n) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(rgb, out, n);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  long w = atol(argv[1]), h = atol(argv[2]);
  long n = w * h;

  // Aligned allocation (16-byte) for SIMD efficiency
  size_t bytes = ((size_t)n * 3 + 15) & ~(size_t)15;
  uint8_t* rgb = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* o_ref = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* o_opt = (uint8_t*)aligned_alloc(16, bytes);
  if (!rgb || !o_ref || !o_opt) {
    printf("Alloc failed\n");
    return 1;
  }

  // A deliberately blue-tinted image so Gray-World has work to do
  for (long i = 0; i < n; i++) {
    rgb[3 * i + 0] = (uint8_t)(90 + ((i * 7) & 0x3F));    // R lower
    rgb[3 * i + 1] = (uint8_t)(110 + ((i * 13) & 0x3F));  // G mid
    rgb[3 * i + 2] = (uint8_t)(160 + ((i * 29) & 0x3F));  // B higher
  }

  // Show the data-dependent gains computed by the Gray-World algorithm
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);

  printf("=======================================================\n");
  printf(" White Balance v1: Gray-World Serial Baseline and Auto-Vec\n");
  printf(" Image: %ld x %ld = %ld px   Loops: %d\n", w, h, n, NTIMES);
  printf(" Channel mean R/G/B = %.1f / %.1f / %.1f\n", (double)sum_r / n,
         (double)sum_g / n, (double)sum_b / n);
  printf(" Computed gains (Q8) = %u / %u / %u  (%.3f / %.3f / %.3f x)\n",
         gain_r, gain_g, gain_b, gain_r / 256.0, gain_g / 256.0,
         gain_b / 256.0);
  printf("=======================================================\n");

  // Golden reference = the fully-serial (no-vec) pipeline
  gray_world_no_vec(rgb, o_ref, n);

  memset(o_opt, 0, bytes);
  double t_nv = bench(gray_world_no_vec, rgb, o_opt, n);
  memset(o_opt, 0, bytes);
  double t_se = bench(gray_world_serial, rgb, o_opt, n);
  const char* s_se = check_diff(o_ref, o_opt, n * 3);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_nv);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_se, t_nv / t_se,
         s_se);
  printf("-------------------------------------------------\n");

  free(rgb);
  free(o_ref);
  free(o_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_whitebalance/whitebalance_v1.c", "src_whitebalance/whitebalance_v1")
out_v1 = run_bin(BIN, 1920, 1080)

## 6. v2 · NEON 两遍实现

在 v1 的基础上，**新增 `gray_world_neon` 实现**，其两遍处理均采用 NEON：

- **第一遍（`sums_neon`）** 采用规约：`vpaddlq_u8` 与 `vpadalq_u16` 逐级加宽累加，`vaddvq_u32` 做水平规约，得到各通道之和（此处对应实验二的规约技术）。
- **第二遍（`apply_neon`）** 采用“解交织—计算—交织”：`vld3` 解交织 → 各通道经 `vmovl`/`vmull_n` 加宽后乘以增益 → `vrshrn` 带舍入窄化 → `vqmovn` 饱和 → `vst3` 交织写回。

### 🔑 知识点
- **为何各通道必须先解交织**：三个通道的增益各不相同，只有拆分为独立通道后才能分别乘以不同的增益，这正是 `vld3`/`vst3` 的价值所在（对照实验三——那里仅用于交换通道，此处则用于分别计算）
- **加宽防溢出**：`c × gain`（增益大于 1）可能超出 u8 乃至 u16 的范围，故须逐级加宽至 u32
- **饱和处理**：`vqmovn` 将超出 255 的结果钳制为 255，无需条件分支

此版本包含**三行**输出。请观察 NEON 相对基准的加速比。

In [ ]:
%%writefile src_whitebalance/whitebalance_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Gray-World gain clamp range in Q8 fixed-point
#define GAIN_MIN 64u    // 0.25x in Q8
#define GAIN_MAX 1024u  // 4.00x in Q8

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference (allow +/-1 rounding difference)
static const char* check_diff(const uint8_t* ref, const uint8_t* test,
                              long bytes) {
  int max_diff = 0;
  for (long i = 0; i < bytes; i++) {
    int d = abs((int)ref[i] - (int)test[i]);
    if (d > max_diff) max_diff = d;
  }
  if (max_diff <= 1)
    return "PASS";
  else
    return "FAIL";
}

// Compute Gray-World gains (Q8) from channel sums, then clamp to [0.25x, 4x]
static inline void gains_from_sums(uint64_t sum_r, uint64_t sum_g,
                                   uint64_t sum_b, uint32_t* gain_r,
                                   uint32_t* gain_g, uint32_t* gain_b) {
  uint64_t total = sum_r + sum_g + sum_b;
  uint32_t r = sum_r ? (uint32_t)((256ULL * total) / (3ULL * sum_r)) : 256u;
  uint32_t g = sum_g ? (uint32_t)((256ULL * total) / (3ULL * sum_g)) : 256u;
  uint32_t b = sum_b ? (uint32_t)((256ULL * total) / (3ULL * sum_b)) : 256u;
  *gain_r = r < GAIN_MIN ? GAIN_MIN : (r > GAIN_MAX ? GAIN_MAX : r);
  *gain_g = g < GAIN_MIN ? GAIN_MIN : (g > GAIN_MAX ? GAIN_MAX : g);
  *gain_b = b < GAIN_MIN ? GAIN_MIN : (b > GAIN_MAX ? GAIN_MAX : b);
}

// Apply gain to one channel sample: (c * gain_q8 + 128) >> 8, clamp to [0,255]
static inline uint8_t wb_px(uint8_t c, uint32_t gain_q8) {
  uint32_t v = ((uint32_t)c * gain_q8 + 128u) >> 8;
  return v > 255u ? 255u : (uint8_t)v;
}

// Pass 1 (serial): sum each channel over the image
static inline void sums_serial(const uint8_t* restrict rgb, long n,
                               uint64_t* sum_r, uint64_t* sum_g,
                               uint64_t* sum_b) {
  uint64_t r = 0, g = 0, b = 0;
  for (long i = 0; i < n; i++) {
    r += rgb[3 * i + 0];
    g += rgb[3 * i + 1];
    b += rgb[3 * i + 2];
  }
  *sum_r = r;
  *sum_g = g;
  *sum_b = b;
}

// Pass 1 (NEON): sum each channel using widening reduction
static inline void sums_neon(const uint8_t* restrict rgb, long n,
                             uint64_t* sum_r, uint64_t* sum_g,
                             uint64_t* sum_b) {
  uint64_t r = 0, g = 0, b = 0;
  uint32x4_t acc_r = vdupq_n_u32(0);
  uint32x4_t acc_g = vdupq_n_u32(0);
  uint32x4_t acc_b = vdupq_n_u32(0);
  long i = 0, cnt = 0;
  for (; i <= n - 16; i += 16) {
    uint8x16x3_t px = vld3q_u8(rgb + i * 3);  // de-interleave R/G/B
    acc_r =
        vpadalq_u16(acc_r, vpaddlq_u8(px.val[0]));  // widen-add + accumulate
    acc_g = vpadalq_u16(acc_g, vpaddlq_u8(px.val[1]));
    acc_b = vpadalq_u16(acc_b, vpaddlq_u8(px.val[2]));
    if (++cnt == 8192) {  // drain to u64 to avoid overflow
      r += vaddvq_u32(acc_r);
      g += vaddvq_u32(acc_g);
      b += vaddvq_u32(acc_b);
      acc_r = vdupq_n_u32(0);
      acc_g = vdupq_n_u32(0);
      acc_b = vdupq_n_u32(0);
      cnt = 0;
    }
  }
  r += vaddvq_u32(acc_r);
  g += vaddvq_u32(acc_g);
  b += vaddvq_u32(acc_b);
  for (; i < n; i++) {  // scalar tail
    r += rgb[3 * i + 0];
    g += rgb[3 * i + 1];
    b += rgb[3 * i + 2];
  }
  *sum_r = r;
  *sum_g = g;
  *sum_b = b;
}

// Pass 2 (serial): apply per-channel gains
static inline void apply_serial(const uint8_t* restrict rgb,
                                uint8_t* restrict out, long n, uint32_t gain_r,
                                uint32_t gain_g, uint32_t gain_b) {
  for (long i = 0; i < n; i++) {
    out[3 * i + 0] = wb_px(rgb[3 * i + 0], gain_r);
    out[3 * i + 1] = wb_px(rgb[3 * i + 1], gain_g);
    out[3 * i + 2] = wb_px(rgb[3 * i + 2], gain_b);
  }
}

// Scale 16 u8 lanes by a Q8 gain: widen -> multiply -> round-narrow -> saturate
static inline uint8x16_t gain_u8x16(uint8x16_t c, uint16_t g) {
  uint16x8_t lo = vmovl_u8(vget_low_u8(c));
  uint16x8_t hi = vmovl_u8(vget_high_u8(c));
  uint16x8_t rlo =
      vcombine_u16(vrshrn_n_u32(vmull_n_u16(vget_low_u16(lo), g), 8),
                   vrshrn_n_u32(vmull_n_u16(vget_high_u16(lo), g), 8));
  uint16x8_t rhi =
      vcombine_u16(vrshrn_n_u32(vmull_n_u16(vget_low_u16(hi), g), 8),
                   vrshrn_n_u32(vmull_n_u16(vget_high_u16(hi), g), 8));
  return vcombine_u8(vqmovn_u16(rlo),
                     vqmovn_u16(rhi));  // vqmovn saturates to 255
}

// Pass 2 (NEON): de-interleave -> scale -> saturate -> interleave
static inline void apply_neon(const uint8_t* restrict rgb,
                              uint8_t* restrict out, long n, uint32_t gain_r,
                              uint32_t gain_g, uint32_t gain_b) {
  long i = 0;
  for (; i <= n - 16; i += 16) {
    uint8x16x3_t px = vld3q_u8(rgb + i * 3);
    px.val[0] = gain_u8x16(px.val[0], (uint16_t)gain_r);
    px.val[1] = gain_u8x16(px.val[1], (uint16_t)gain_g);
    px.val[2] = gain_u8x16(px.val[2], (uint16_t)gain_b);
    vst3q_u8(out + i * 3, px);
  }
  for (; i < n; i++) {  // scalar tail
    out[3 * i + 0] = wb_px(rgb[3 * i + 0], gain_r);
    out[3 * i + 1] = wb_px(rgb[3 * i + 1], gain_g);
    out[3 * i + 2] = wb_px(rgb[3 * i + 2], gain_b);
  }
}

// Full pipeline: Serial statistics + Serial apply (compiler may auto-vectorize)
void gray_world_serial(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_serial(rgb, out, n, gain_r, gain_g, gain_b);
}

// Full pipeline: Serial statistics + Serial apply (vectorization disabled) -
// Baseline
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gray_world_no_vec(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_serial(rgb, out, n, gain_r, gain_g, gain_b);
}

// Full pipeline: NEON statistics + NEON apply
void gray_world_neon(const uint8_t* restrict rgb, uint8_t* restrict out,
                     long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_neon(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_neon(rgb, out, n, gain_r, gain_g, gain_b);
}

// Benchmark helper: run kernel NTIMES, keep the best (lowest) time
typedef void (*wb_fn)(const uint8_t*, uint8_t*, long);

static double bench(wb_fn fn, const uint8_t* rgb, uint8_t* out, long n) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(rgb, out, n);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  long w = atol(argv[1]), h = atol(argv[2]);
  long n = w * h;

  // Aligned allocation (16-byte) for SIMD efficiency
  size_t bytes = ((size_t)n * 3 + 15) & ~(size_t)15;
  uint8_t* rgb = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* o_ref = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* o_opt = (uint8_t*)aligned_alloc(16, bytes);
  if (!rgb || !o_ref || !o_opt) {
    printf("Alloc failed\n");
    return 1;
  }

  // A deliberately blue-tinted image so Gray-World has work to do
  for (long i = 0; i < n; i++) {
    rgb[3 * i + 0] = (uint8_t)(90 + ((i * 7) & 0x3F));    // R lower
    rgb[3 * i + 1] = (uint8_t)(110 + ((i * 13) & 0x3F));  // G mid
    rgb[3 * i + 2] = (uint8_t)(160 + ((i * 29) & 0x3F));  // B higher
  }

  // Show the data-dependent gains computed by the Gray-World algorithm
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);

  printf("=======================================================\n");
  printf(" White Balance v2: Add NEON Reduction and Apply\n");
  printf(" Image: %ld x %ld = %ld px   Loops: %d\n", w, h, n, NTIMES);
  printf(" Channel mean R/G/B = %.1f / %.1f / %.1f\n", (double)sum_r / n,
         (double)sum_g / n, (double)sum_b / n);
  printf(" Computed gains (Q8) = %u / %u / %u  (%.3f / %.3f / %.3f x)\n",
         gain_r, gain_g, gain_b, gain_r / 256.0, gain_g / 256.0,
         gain_b / 256.0);
  printf("=======================================================\n");

  // Golden reference = the fully-serial (no-vec) pipeline
  gray_world_no_vec(rgb, o_ref, n);

  memset(o_opt, 0, bytes);
  double t_nv = bench(gray_world_no_vec, rgb, o_opt, n);
  memset(o_opt, 0, bytes);
  double t_se = bench(gray_world_serial, rgb, o_opt, n);
  const char* s_se = check_diff(o_ref, o_opt, n * 3);
  memset(o_opt, 0, bytes);
  double t_ne = bench(gray_world_neon, rgb, o_opt, n);
  const char* s_ne = check_diff(o_ref, o_opt, n * 3);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_nv);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_se, t_nv / t_se,
         s_se);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_ne, t_nv / t_ne,
         s_ne);
  printf("-------------------------------------------------\n");

  free(rgb);
  free(o_ref);
  free(o_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_whitebalance/whitebalance_v2.c", "src_whitebalance/whitebalance_v2")
out_v2 = run_bin(BIN, 1920, 1080)

## 7. v3 · 应用阶段循环展开

在 v2 的基础上，**新增 `gray_world_unroll` 实现**：第一遍规约统计保持不变，第二遍应用阶段改用展开版本 `apply_neon_unroll`，单次迭代处理 32 个像素（两个 16 像素块）。

### 🔑 知识点
- **在应用阶段展开**：应用阶段是逐像素的独立运算，展开可减少循环控制开销、提高访存与计算的重叠度
- **统计阶段不展开**：第一遍只读 3n 字节、且已是很紧凑的规约（每 16 像素约 12 条指令），本身受访存制约；第二遍要读 3n、写 3n，指令链也长得多（每 16 像素约 78 条），才是值得优化的

> 白平衡需要对图像进行**两遍**遍历（先统计、后应用），其中第一遍受访存制约。但第二遍并非如此——它还留有指令级并行的余地，因此展开在这里能带来性能提升。

至此，**四个实现**（含基准）全部实现。

In [ ]:
%%writefile src_whitebalance/whitebalance_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Gray-World gain clamp range in Q8 fixed-point
#define GAIN_MIN 64u    // 0.25x in Q8
#define GAIN_MAX 1024u  // 4.00x in Q8

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference (allow +/-1 rounding difference)
static const char* check_diff(const uint8_t* ref, const uint8_t* test,
                              long bytes) {
  int max_diff = 0;
  for (long i = 0; i < bytes; i++) {
    int d = abs((int)ref[i] - (int)test[i]);
    if (d > max_diff) max_diff = d;
  }
  if (max_diff <= 1)
    return "PASS";
  else
    return "FAIL";
}

// Compute Gray-World gains (Q8) from channel sums, then clamp to [0.25x, 4x]
static inline void gains_from_sums(uint64_t sum_r, uint64_t sum_g,
                                   uint64_t sum_b, uint32_t* gain_r,
                                   uint32_t* gain_g, uint32_t* gain_b) {
  uint64_t total = sum_r + sum_g + sum_b;
  uint32_t r = sum_r ? (uint32_t)((256ULL * total) / (3ULL * sum_r)) : 256u;
  uint32_t g = sum_g ? (uint32_t)((256ULL * total) / (3ULL * sum_g)) : 256u;
  uint32_t b = sum_b ? (uint32_t)((256ULL * total) / (3ULL * sum_b)) : 256u;
  *gain_r = r < GAIN_MIN ? GAIN_MIN : (r > GAIN_MAX ? GAIN_MAX : r);
  *gain_g = g < GAIN_MIN ? GAIN_MIN : (g > GAIN_MAX ? GAIN_MAX : g);
  *gain_b = b < GAIN_MIN ? GAIN_MIN : (b > GAIN_MAX ? GAIN_MAX : b);
}

// Apply gain to one channel sample: (c * gain_q8 + 128) >> 8, clamp to [0,255]
static inline uint8_t wb_px(uint8_t c, uint32_t gain_q8) {
  uint32_t v = ((uint32_t)c * gain_q8 + 128u) >> 8;
  return v > 255u ? 255u : (uint8_t)v;
}

// Pass 1 (serial): sum each channel over the image
static inline void sums_serial(const uint8_t* restrict rgb, long n,
                               uint64_t* sum_r, uint64_t* sum_g,
                               uint64_t* sum_b) {
  uint64_t r = 0, g = 0, b = 0;
  for (long i = 0; i < n; i++) {
    r += rgb[3 * i + 0];
    g += rgb[3 * i + 1];
    b += rgb[3 * i + 2];
  }
  *sum_r = r;
  *sum_g = g;
  *sum_b = b;
}

// Pass 1 (NEON): sum each channel using widening reduction
static inline void sums_neon(const uint8_t* restrict rgb, long n,
                             uint64_t* sum_r, uint64_t* sum_g,
                             uint64_t* sum_b) {
  uint64_t r = 0, g = 0, b = 0;
  uint32x4_t acc_r = vdupq_n_u32(0);
  uint32x4_t acc_g = vdupq_n_u32(0);
  uint32x4_t acc_b = vdupq_n_u32(0);
  long i = 0, cnt = 0;
  for (; i <= n - 16; i += 16) {
    uint8x16x3_t px = vld3q_u8(rgb + i * 3);  // de-interleave R/G/B
    acc_r =
        vpadalq_u16(acc_r, vpaddlq_u8(px.val[0]));  // widen-add + accumulate
    acc_g = vpadalq_u16(acc_g, vpaddlq_u8(px.val[1]));
    acc_b = vpadalq_u16(acc_b, vpaddlq_u8(px.val[2]));
    if (++cnt == 8192) {  // drain to u64 to avoid overflow
      r += vaddvq_u32(acc_r);
      g += vaddvq_u32(acc_g);
      b += vaddvq_u32(acc_b);
      acc_r = vdupq_n_u32(0);
      acc_g = vdupq_n_u32(0);
      acc_b = vdupq_n_u32(0);
      cnt = 0;
    }
  }
  r += vaddvq_u32(acc_r);
  g += vaddvq_u32(acc_g);
  b += vaddvq_u32(acc_b);
  for (; i < n; i++) {  // scalar tail
    r += rgb[3 * i + 0];
    g += rgb[3 * i + 1];
    b += rgb[3 * i + 2];
  }
  *sum_r = r;
  *sum_g = g;
  *sum_b = b;
}

// Pass 2 (serial): apply per-channel gains
static inline void apply_serial(const uint8_t* restrict rgb,
                                uint8_t* restrict out, long n, uint32_t gain_r,
                                uint32_t gain_g, uint32_t gain_b) {
  for (long i = 0; i < n; i++) {
    out[3 * i + 0] = wb_px(rgb[3 * i + 0], gain_r);
    out[3 * i + 1] = wb_px(rgb[3 * i + 1], gain_g);
    out[3 * i + 2] = wb_px(rgb[3 * i + 2], gain_b);
  }
}

// Scale 16 u8 lanes by a Q8 gain: widen -> multiply -> round-narrow -> saturate
static inline uint8x16_t gain_u8x16(uint8x16_t c, uint16_t g) {
  uint16x8_t lo = vmovl_u8(vget_low_u8(c));
  uint16x8_t hi = vmovl_u8(vget_high_u8(c));
  uint16x8_t rlo =
      vcombine_u16(vrshrn_n_u32(vmull_n_u16(vget_low_u16(lo), g), 8),
                   vrshrn_n_u32(vmull_n_u16(vget_high_u16(lo), g), 8));
  uint16x8_t rhi =
      vcombine_u16(vrshrn_n_u32(vmull_n_u16(vget_low_u16(hi), g), 8),
                   vrshrn_n_u32(vmull_n_u16(vget_high_u16(hi), g), 8));
  return vcombine_u8(vqmovn_u16(rlo),
                     vqmovn_u16(rhi));  // vqmovn saturates to 255
}

// Pass 2 (NEON): de-interleave -> scale -> saturate -> interleave
static inline void apply_neon(const uint8_t* restrict rgb,
                              uint8_t* restrict out, long n, uint32_t gain_r,
                              uint32_t gain_g, uint32_t gain_b) {
  long i = 0;
  for (; i <= n - 16; i += 16) {
    uint8x16x3_t px = vld3q_u8(rgb + i * 3);
    px.val[0] = gain_u8x16(px.val[0], (uint16_t)gain_r);
    px.val[1] = gain_u8x16(px.val[1], (uint16_t)gain_g);
    px.val[2] = gain_u8x16(px.val[2], (uint16_t)gain_b);
    vst3q_u8(out + i * 3, px);
  }
  for (; i < n; i++) {  // scalar tail
    out[3 * i + 0] = wb_px(rgb[3 * i + 0], gain_r);
    out[3 * i + 1] = wb_px(rgb[3 * i + 1], gain_g);
    out[3 * i + 2] = wb_px(rgb[3 * i + 2], gain_b);
  }
}

// Pass 2 (NEON, unrolled): process 32 pixels per iteration
static inline void apply_neon_unroll(const uint8_t* restrict rgb,
                                     uint8_t* restrict out, long n,
                                     uint32_t gain_r, uint32_t gain_g,
                                     uint32_t gain_b) {
  long i = 0;
  for (; i <= n - 32; i += 32) {
    uint8x16x3_t p0 = vld3q_u8(rgb + i * 3);
    uint8x16x3_t p1 = vld3q_u8(rgb + (i + 16) * 3);
    p0.val[0] = gain_u8x16(p0.val[0], (uint16_t)gain_r);
    p0.val[1] = gain_u8x16(p0.val[1], (uint16_t)gain_g);
    p0.val[2] = gain_u8x16(p0.val[2], (uint16_t)gain_b);
    p1.val[0] = gain_u8x16(p1.val[0], (uint16_t)gain_r);
    p1.val[1] = gain_u8x16(p1.val[1], (uint16_t)gain_g);
    p1.val[2] = gain_u8x16(p1.val[2], (uint16_t)gain_b);
    vst3q_u8(out + i * 3, p0);
    vst3q_u8(out + (i + 16) * 3, p1);
  }
  for (; i <= n - 16; i += 16) {
    uint8x16x3_t px = vld3q_u8(rgb + i * 3);
    px.val[0] = gain_u8x16(px.val[0], (uint16_t)gain_r);
    px.val[1] = gain_u8x16(px.val[1], (uint16_t)gain_g);
    px.val[2] = gain_u8x16(px.val[2], (uint16_t)gain_b);
    vst3q_u8(out + i * 3, px);
  }
  for (; i < n; i++) {  // scalar tail
    out[3 * i + 0] = wb_px(rgb[3 * i + 0], gain_r);
    out[3 * i + 1] = wb_px(rgb[3 * i + 1], gain_g);
    out[3 * i + 2] = wb_px(rgb[3 * i + 2], gain_b);
  }
}

// Full pipeline: Serial statistics + Serial apply (compiler may auto-vectorize)
void gray_world_serial(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_serial(rgb, out, n, gain_r, gain_g, gain_b);
}

// Full pipeline: Serial statistics + Serial apply (vectorization disabled) -
// Baseline
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gray_world_no_vec(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_serial(rgb, out, n, gain_r, gain_g, gain_b);
}

// Full pipeline: NEON statistics + NEON apply
void gray_world_neon(const uint8_t* restrict rgb, uint8_t* restrict out,
                     long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_neon(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_neon(rgb, out, n, gain_r, gain_g, gain_b);
}

// Full pipeline: NEON statistics + NEON unrolled apply
void gray_world_unroll(const uint8_t* restrict rgb, uint8_t* restrict out,
                       long n) {
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_neon(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);
  apply_neon_unroll(rgb, out, n, gain_r, gain_g, gain_b);
}

// Benchmark helper: run kernel NTIMES, keep the best (lowest) time
typedef void (*wb_fn)(const uint8_t*, uint8_t*, long);

static double bench(wb_fn fn, const uint8_t* rgb, uint8_t* out, long n) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(rgb, out, n);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  long w = atol(argv[1]), h = atol(argv[2]);
  long n = w * h;

  // Aligned allocation (16-byte) for SIMD efficiency
  size_t bytes = ((size_t)n * 3 + 15) & ~(size_t)15;
  uint8_t* rgb = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* o_ref = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* o_opt = (uint8_t*)aligned_alloc(16, bytes);
  if (!rgb || !o_ref || !o_opt) {
    printf("Alloc failed\n");
    return 1;
  }

  // A deliberately blue-tinted image so Gray-World has work to do
  for (long i = 0; i < n; i++) {
    rgb[3 * i + 0] = (uint8_t)(90 + ((i * 7) & 0x3F));    // R lower
    rgb[3 * i + 1] = (uint8_t)(110 + ((i * 13) & 0x3F));  // G mid
    rgb[3 * i + 2] = (uint8_t)(160 + ((i * 29) & 0x3F));  // B higher
  }

  // Show the data-dependent gains computed by the Gray-World algorithm
  uint64_t sum_r, sum_g, sum_b;
  uint32_t gain_r, gain_g, gain_b;
  sums_serial(rgb, n, &sum_r, &sum_g, &sum_b);
  gains_from_sums(sum_r, sum_g, sum_b, &gain_r, &gain_g, &gain_b);

  printf("=======================================================\n");
  printf(" White Balance v3: Add NEON Unrolled Apply\n");
  printf(" Image: %ld x %ld = %ld px   Loops: %d\n", w, h, n, NTIMES);
  printf(" Channel mean R/G/B = %.1f / %.1f / %.1f\n", (double)sum_r / n,
         (double)sum_g / n, (double)sum_b / n);
  printf(" Computed gains (Q8) = %u / %u / %u  (%.3f / %.3f / %.3f x)\n",
         gain_r, gain_g, gain_b, gain_r / 256.0, gain_g / 256.0,
         gain_b / 256.0);
  printf("=======================================================\n");

  // Golden reference = the fully-serial (no-vec) pipeline
  gray_world_no_vec(rgb, o_ref, n);

  memset(o_opt, 0, bytes);
  double t_nv = bench(gray_world_no_vec, rgb, o_opt, n);
  memset(o_opt, 0, bytes);
  double t_se = bench(gray_world_serial, rgb, o_opt, n);
  const char* s_se = check_diff(o_ref, o_opt, n * 3);
  memset(o_opt, 0, bytes);
  double t_ne = bench(gray_world_neon, rgb, o_opt, n);
  const char* s_ne = check_diff(o_ref, o_opt, n * 3);
  memset(o_opt, 0, bytes);
  double t_un = bench(gray_world_unroll, rgb, o_opt, n);
  const char* s_un = check_diff(o_ref, o_opt, n * 3);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_nv);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_se, t_nv / t_se,
         s_se);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_ne, t_nv / t_ne,
         s_ne);
  printf("| NEON Unrolled   | %9.3f | %5.2f x |  %-4s |\n", t_un, t_nv / t_un,
         s_un);
  printf("-------------------------------------------------\n");

  free(rgb);
  free(o_ref);
  free(o_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_whitebalance/whitebalance_v3.c", "src_whitebalance/whitebalance_v3")
out_v3 = run_bin(BIN, 1920, 1080)

## 8. 📈 性能可视化（基于 v3 的四版本结果）

v3 的输出包含全部四个实现的耗时与加速比，据此绘制柱状图，以完整呈现各手段的效果。
（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

In [ ]:
rows_v3 = parse_table(out_v3)
for r in rows_v3:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:.2f}x')
plot_speedup(rows_v3, "White Balance (Gray World): performance of four implementations (1920x1080)")

## 9. 结果分析

> 注：具体数值随硬件平台、图像尺寸、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

**① 首先验证增益的数据相关性。**

程序打印的通道均值为 R/G/B = 121.5 / 141.5 / 191.5（明显偏蓝），算出的增益为 $1.246\times$ / $1.070\times$ / $0.789\times$——R 被放大、B 被压暗，正是 Gray World 期望的方向。这表明增益确实是**由图像统计量算出**的，而非预先设定。

**② 编译器自动向量化在本例中已相当有效，手写 NEON 的优势主要来自循环展开。**

实测中 `Serial (Auto)` 与 `NEON Intrinsic` 性能接近，二者相对基准均达 3× 左右；进一步展开后（`NEON Unrolled`）有进一步的提升。

其原因在于：白平衡的两遍处理中，**统计**与**应用**均为规整的逐像素循环，编译器有能力对其自动向量化。

**③ 循环展开是本例中收益最明显的一步。**

从 `NEON Intrinsic`到 `NEON Unrolled`，性能提升约 10%，而此前 `Serial (Auto)` 到 `NEON Intrinsic` 几乎没有差距。这说明：在编译器已能有效向量化的前提下，手写 NEON 的额外价值主要体现在**更激进的循环展开**上——它减少了循环控制开销，并提高了访存与计算的重叠度。

需要注意的是，白平衡需要对图像进行**两遍**遍历（统计一遍、应用一遍），数据被读取两次，整体仍受**访存**制约，因此各版本的加速比都被限制在一定的范围内，未能随向量宽度线性增长。

---

### 🎓 结论
白平衡是本章的第一个**综合案例**，它将**规约 + 结构化访存 + 定点运算 + 加宽防溢出 + 饱和处理**整合于一个真实、数据相关的算法之中。

本实验给出了一个值得注意的结论：**并非所有含规约的运算都难以自动向量化。** 白平衡的统计阶段是**整数**累加，整数加法满足结合律，编译器可以安全地重排求和顺序并将其向量化；而实验二 GEMV 的内层是**浮点**规约，在未启用 `-ffast-math` 时编译器不能改变累加顺序，因此自动向量化失效。这正是二者 `Serial (Auto)` 表现迥异的原因。

在编译器已能胜任的场景下，手写 NEON 的价值主要体现在更激进的展开与调度上。

## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察结果的变化（建议先独立完成，再阅读思考题）：

1. 将增益的钳制范围由 `[0.25×, 4×]`（即 `GAIN_MIN`/`GAIN_MAX`）改窄（如 `[0.5×, 2×]`，对应 Q8 整数 `[128, 512]`），观察其对输出结果与数值稳定性的影响。
2. 修改 `main` 中测试图像的初始化，使其偏黄（R、G 较高、B 较低），观察算出的增益方向是否符合预期。
3. 尝试仅统计 1/16 的像素来估计通道均值（修改第一遍的循环步长），比较其对速度与结果的影响。
4. 在 `main` 中为 `sums_neon` 与 `apply_neon` 分别加一段计时，看看两遍各占多少时间。
5. 【进阶】为 `compile_c` 的编译参数添加 `-march=native` 后重新运行，观察 `Serial (Auto)` 是否有所改善，以及是否仍明显低于手写 NEON。

## 11. 🤔 思考题

- 为何白平衡必须采用“两遍”处理？能否合并为一遍完成？合并的代价是什么？
- 增益公式 `gain=(ΣR+ΣG+ΣB)/(3·Σc)` 为何可以不先除以像素数 N？
- 为何采用 Q8 定点而非浮点来表示与计算增益？在边缘端设备上各有何利弊？
- 若改用浮点实现，增益应钳制于什么范围？它与本实验的整数区间 `[64, 1024]` 是何关系？

## 12. 小结与后续

本实验完成了 Gray World 白平衡从全串行到 NEON 的实现过程：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>gray_world_no_vec</code> + <code>gray_world_serial</code></td>
      <td style="text-align: left;">性能基准、数据相关增益、两遍法、Q8 定点</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>gray_world_neon</code></td>
      <td style="text-align: left;">规约统计、结构化访存、加宽防溢出、饱和处理</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>gray_world_unroll</code></td>
      <td style="text-align: left;">应用阶段循环展开</td>
    </tr>
  </tbody>
</table>

白平衡将**规约、结构化访存、定点运算、加宽与饱和**整合于一个真实的数据相关算法之中，是本章第一个综合案例。它给出了一条重要认识：**含规约的运算未必难以自动向量化——关键在于该依赖能否被编译器安全打破**。在编译器已能有效向量化的场景下，手写 NEON 的价值主要体现在更激进的循环展开与调度上。

➡️ **后续内容：实验五 RGB→Gray**。我们将把三个通道**加权规约为单个灰度平面**，它与本实验形成一组有意思的对照：两者的自动向量化都已足够好，但实验五连**循环展开也拿不到收益**。